### 1 - Extract e diagnóstico (tipos, ausentes, duplicatas, categorias)

Vamos carregar o arquivo de exames laboratoriais e fazer um diagnóstico inicial, verificando: o formato dos dados (shape), os tipos de cada coluna (dtypes), a 
quantidade de valores ausentes por coluna, a quantidade de linhas duplicadas, e a distribuição das categorias (sexo, prioridade, laboratorio, tipo_exame) para 
identificar inconsistências de grafia antes de qualquer correção.

In [1]:
import pandas as pd
import numpy as np

ex = pd.read_csv("dados/exames_laboratoriais.csv")

print("Formato:", ex.shape)
print("\nTipos de cada coluna:")
print(ex.dtypes)
print("\nValores ausentes por coluna:")
print(ex.isna().sum()[lambda s: s > 0])
print("\nLinhas duplicadas:", ex.duplicated().sum())
print("\nCategorias:")
for col in ["sexo", "prioridade", "laboratorio", "tipo_exame"]:
    print(f"\n{col} →", ex[col].value_counts(dropna=False).to_dict())

Formato: (155, 10)

Tipos de cada coluna:
id_exame        object
id_paciente     object
data_coleta     object
sexo            object
idade          float64
tipo_exame      object
resultado       object
unidade         object
laboratorio     object
prioridade      object
dtype: object

Valores ausentes por coluna:
idade    7
dtype: int64

Linhas duplicadas: 5

Categorias:

sexo → {'M': 72, 'F': 69, 'masculino': 10, 'Feminino': 4}

prioridade → {'Rotina': 97, 'Urgente': 36, 'ROTINA': 11, 'Emergência': 9, 'URGENTE': 1, 'EMERGÊNCIA': 1}

laboratorio → {'LabSaúde RN': 53, 'Laboratório Central': 49, 'LabNatal': 42, ' labnatal ': 6, ' labsaúde rn ': 5}

tipo_exame → {'Colesterol total': 46, 'Glicemia': 46, 'Creatinina': 39, 'Hemoglobina': 24}


### 2 - Remoção de duplicatas

Vamos remover as 5 linhas duplicadas identificadas no diagnóstico, resetar o índice e conferir que id_exame (o identificador único de cada exame) ficou sem repetições.

In [2]:
antes = len(ex)
ex = ex.drop_duplicates().reset_index(drop=True)
print(f"Linhas antes: {antes} | depois: {len(ex)}")
print("id_exame é único?", ex["id_exame"].is_unique)

Linhas antes: 155 | depois: 150
id_exame é único? True


### 3 - Transformação de tipos

Vamos converter data_coleta (que vem em dois formatos, ISO e brasileiro) para o tipo datetime, e resultado (texto com vírgula decimal, ex: "84,2") para float, para que 
ambas as colunas possam ser usadas em cálculos e comparações.

In [3]:
# data_coleta: dois formatos misturados (igual à Aula 09/10)
iso = pd.to_datetime(ex["data_coleta"], format="%Y-%m-%d", errors="coerce")
br = pd.to_datetime(ex["data_coleta"], format="%d/%m/%Y", errors="coerce")
ex["data_coleta"] = iso.fillna(br)

print("Datas não convertidas:", ex["data_coleta"].isna().sum())
print("Período:", ex["data_coleta"].min().date(), "a", ex["data_coleta"].max().date())

# resultado: texto com vírgula decimal (e o código 999, que trataremos na Seção 5)
ex["resultado"] = ex["resultado"].str.replace(",", ".", regex=False).astype(float)

print("\nTipo de 'resultado':", ex["resultado"].dtype)
print(ex["resultado"].describe().round(2))

Datas não convertidas: 0
Período: 2026-07-02 a 2026-09-28

Tipo de 'resultado': float64
count    150.00
mean     112.05
std      167.42
min        0.47
25%       12.42
50%       87.05
75%      171.48
max      999.00
Name: resultado, dtype: float64


### 4 - Padronização das categorias

Vamos padronizar as três colunas categóricas que apareceram com grafias inconsistentes no diagnóstico: sexo (M, F, masculino, Feminino -> F/M), prioridade (ROTINA, URGENTE,
EMERGÊNCIA -> Rotina/Urgente/Emergência) e laboratorio (espaços extras e minúsculas -> nomes oficiais LabNatal, LabSaúde RN e Laboratório Central).

In [4]:
# sexo: primeira letra maiúscula (masculino -> M, Feminino -> F)
ex["sexo"] = ex["sexo"].str.strip().str[0].str.upper()

# prioridade: tira espaços e deixa só a primeira letra maiúscula (ROTINA -> Rotina)
ex["prioridade"] = ex["prioridade"].str.strip().str.capitalize()

# laboratorio: tira espaços, uniformiza em minúsculas e mapeia para o nome oficial
nomes_oficiais = {
    "labnatal": "LabNatal",
    "labsaúde rn": "LabSaúde RN",
    "laboratório central": "Laboratório Central",
}
ex["laboratorio"] = ex["laboratorio"].str.strip().str.lower().map(nomes_oficiais)

for col in ["sexo", "prioridade", "laboratorio"]:
    print(f"{col} →", ex[col].value_counts(dropna=False).to_dict())

sexo → {'M': 80, 'F': 70}
prioridade → {'Rotina': 105, 'Urgente': 35, 'Emergência': 10}
laboratorio → {'LabSaúde RN': 58, 'Laboratório Central': 48, 'LabNatal': 44}


### 5 - Valores inválidos e ausentes

O valor 999 em `resultado` é um código sentinela que significa "amostra não processada" e não uma medida real, então será convertido em ausente (NaN). Depois, os ausentes de
`resultado` serão imputados pela mediana do mesmo tipo de exame, porque glicemia, colesterol, creatinina e hemoglobina têm escalas completamente diferentes, e os ausentes de `idade` serão imputados pela mediana geral. A ordem importa: primeiro convertemos o 999 em NaN e só depois calculamos as medianas, para que o código não contamine a imputação.

In [5]:
# Diagnóstico antes de mexer
print("Resultados com código 999:", (ex["resultado"] == 999).sum())
print("Idades ausentes:", ex["idade"].isna().sum())
print("Média de 'resultado' ANTES:", round(ex["resultado"].mean(), 2))

# 1) código sentinela -> ausente
ex["resultado"] = ex["resultado"].replace(999, np.nan)
print("Média de 'resultado' DEPOIS de trocar 999 por NaN:", round(ex["resultado"].mean(), 2))

# 2) imputação
ex["resultado"] = ex["resultado"].fillna(ex.groupby("tipo_exame")["resultado"].transform("median"))
ex["idade"] = ex["idade"].fillna(ex["idade"].median())

# Conferência
print("\nAusentes restantes:", int(ex.isna().sum().sum()))
print("\nMediana de 'resultado' por tipo de exame:")
print(ex.groupby("tipo_exame")["resultado"].median().round(2))

Resultados com código 999: 4
Idades ausentes: 6
Média de 'resultado' ANTES: 112.05
Média de 'resultado' DEPOIS de trocar 999 por NaN: 87.75

Ausentes restantes: 0

Mediana de 'resultado' por tipo de exame:
tipo_exame
Colesterol total    193.80
Creatinina            0.92
Glicemia             94.25
Hemoglobina          13.80
Name: resultado, dtype: float64


### 6 - Engenharia de atributos

Vamos criar três atributos novos a partir das colunas existentes: `faixa_etaria` (idade discretizada com `pd.cut` nas faixas 18-39, 40-59, 60-79 e 80+), `mes_coleta`
(mês extraído da data da coleta) e `alterado` (True quando o resultado está fora da faixa de referência do respectivo tipo de exame). Esses atributos transformam valores contínuos e datas em informações mais fáceis de analisar.

In [6]:
# 1) faixa_etaria: limites escolhidos por nós (pd.cut)
limites = [18, 40, 60, 80, 120]
rotulos = ["18-39", "40-59", "60-79", "80+"]
ex["faixa_etaria"] = pd.cut(ex["idade"], bins=limites, labels=rotulos, right=False)

# 2) mes_coleta: mês (1-12) extraído da data
ex["mes_coleta"] = ex["data_coleta"].dt.month

# 3) alterado: fora da faixa de referência do próprio tipo de exame
res, tipo = ex["resultado"], ex["tipo_exame"]
ex["alterado"] = (
    ((tipo == "Glicemia")         & ((res < 70)  | (res > 99)))   |
    ((tipo == "Colesterol total") & (res >= 190))                 |
    ((tipo == "Creatinina")       & ((res < 0.6) | (res > 1.2)))  |
    ((tipo == "Hemoglobina")      & ((res < 12.0) | (res > 17.5)))
)

print(ex["faixa_etaria"].value_counts().sort_index().to_string())
print("\nMeses:", sorted(ex["mes_coleta"].unique()))
print("\nProporção de exames alterados por tipo:")
print(ex.groupby("tipo_exame")["alterado"].mean().round(2).to_string())
print("\nNulos nas colunas novas:", int(ex[["faixa_etaria", "mes_coleta", "alterado"]].isna().sum().sum()))

faixa_etaria
18-39    47
40-59    41
60-79    44
80+      18

Meses: [np.int32(7), np.int32(8), np.int32(9)]

Proporção de exames alterados por tipo:
tipo_exame
Colesterol total    0.58
Creatinina          0.14
Glicemia            0.36
Hemoglobina         0.04

Nulos nas colunas novas: 0


### 7 - Codificação das categorias

Vamos transformar categorias em números. `prioridade` é uma variável **ordinal** (existe uma ordem real: Rotina < Urgente < Emergência), então usamos um dicionário explícito
para criar `prioridade_cod` (0, 1, 2). Já `tipo_exame` é **nominal** (não existe ordem entre glicemia, colesterol, creatinina e hemoglobina), então usamos one-hot encoding,
que cria uma coluna 0/1 para cada tipo, sem sugerir uma ordem que não existe.

In [7]:
# 1) Ordinal: a ordem é real, então informamos a ordem com um dicionário
ordem_prioridade = {"Rotina": 0, "Urgente": 1, "Emergência": 2}
ex["prioridade_cod"] = ex["prioridade"].map(ordem_prioridade)

# 2) Nominal: one-hot (uma coluna 0/1 por tipo de exame)
onehot = pd.get_dummies(ex["tipo_exame"], prefix="tipo", dtype=int)
ex = pd.concat([ex, onehot], axis=1)

print(ex[["prioridade", "prioridade_cod"]].drop_duplicates().sort_values("prioridade_cod").to_string(index=False))
print("\nColunas one-hot criadas:", list(onehot.columns))
print("Cada linha tem exatamente um 1?", (onehot.sum(axis=1) == 1).all())
print("Nulos em prioridade_cod:", ex["prioridade_cod"].isna().sum())
print("Formato atual:", ex.shape)

prioridade  prioridade_cod
    Rotina               0
   Urgente               1
Emergência               2

Colunas one-hot criadas: ['tipo_Colesterol total', 'tipo_Creatinina', 'tipo_Glicemia', 'tipo_Hemoglobina']
Cada linha tem exatamente um 1? True
Nulos em prioridade_cod: 0
Formato atual: (150, 18)
